### Подключение библиотек

In [1]:
from bs4 import BeautifulSoup as bs

In [2]:
import requests

In [2]:
import pandas as pd

### Получение информаций

In [4]:
url = 'https://w140.zona.plus/movies/' 
page = requests.get(url)

In [5]:
page.status_code

200

In [6]:
soup = bs(page.text, 'html.parser')

In [7]:
page.text

'<!DOCTYPE html>\n<html lang="ru">\n<head>\n  <meta charset="utf-8">\n<meta name="viewport" content="width=device-width,initial-scale=1,maximum-scale=1,user-scalable=no">\n<meta name="mobile-web-app-capable" content="yes">\n<meta name="apple-mobile-web-app-capable" content="yes">\n<link rel="icon" type="image/png" href="/favicon-192x192.png" sizes="192x192">\n<link rel="icon" type="image/png" href="/favicon-16x16.png" sizes="16x16">\n<link rel="icon" type="image/png" href="/favicon-32x32.png" sizes="32x32">\n<link rel="apple-touch-icon" href="/apple-touch-icon.png">\n<link rel="apple-touch-icon" sizes="152x152" href="/apple-touch-icon-152x152.png">\n    <link rel="canonical" href="https://w140.zona.plus/movies">\n<meta name="msapplication-TileColor" content="#2d89ef">\n<meta name="msapplication-TileImage" content="/mstile-144x144.png">\n<title>zona.plus (ex zona.mobi) - смотреть фильмы 2026 онлайн, смотреть новинки в хорошем качестве</title>\n<meta name="description" content="Смотреть 

In [8]:
soup

<!DOCTYPE html>

<html lang="ru">
<head>
<meta charset="utf-8"/>
<meta content="width=device-width,initial-scale=1,maximum-scale=1,user-scalable=no" name="viewport"/>
<meta content="yes" name="mobile-web-app-capable"/>
<meta content="yes" name="apple-mobile-web-app-capable"/>
<link href="/favicon-192x192.png" rel="icon" sizes="192x192" type="image/png"/>
<link href="/favicon-16x16.png" rel="icon" sizes="16x16" type="image/png"/>
<link href="/favicon-32x32.png" rel="icon" sizes="32x32" type="image/png"/>
<link href="/apple-touch-icon.png" rel="apple-touch-icon"/>
<link href="/apple-touch-icon-152x152.png" rel="apple-touch-icon" sizes="152x152"/>
<link href="https://w140.zona.plus/movies" rel="canonical"/>
<meta content="#2d89ef" name="msapplication-TileColor"/>
<meta content="/mstile-144x144.png" name="msapplication-TileImage"/>
<title>zona.plus (ex zona.mobi) - смотреть фильмы 2026 онлайн, смотреть новинки в хорошем качестве</title>
<meta content="Смотреть новинки 2026 года в хорошем к

In [9]:
result_list = {'title': [], 'genre': [], 'year': [], 'country': [], 'rating': [], 'description': []}

### Алгоритм

In [10]:
import time
import random

base_url = 'https://w140.zona.plus'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

session = requests.Session()
session.headers.update(headers)

pagenum = 1
max_movies = 250 
count = 0

while count < max_movies:
    url = f'{base_url}/movies?page={pagenum}'
    try:
        response = session.get(url, timeout=15)
        soup = bs(response.text, 'html.parser')
        movie_links = soup.find_all('a', class_='results-item')
        if not movie_links: break

        for movie in movie_links:
            if count >= max_movies: break
            
            title = movie.find('div', class_='results-item-title').text.strip()
            year = movie.find('span', class_='results-item-year').text.strip()
            rating = movie.find('span', class_='results-item-rating').text.strip() if movie.find('span', class_='results-item-rating') else "0.0"
            movie_url = base_url + movie.get('href')

            try:
                m_res = session.get(movie_url, timeout=10)
                m_soup = bs(m_res.text, 'html.parser')

                movie_genre = "Не указан"
                movie_country = "Не указана"

                dts = m_soup.find_all('dt')
                for dt in dts:
                    label = dt.text.strip()
                    dd = dt.find_next_sibling('dd')
                    if dd:
                        clean_value = ' '.join(dd.text.replace('\n', ' ').split())
                        
                        if "Жанр" in label:
                            movie_genre = clean_value
                        elif "Страна" in label:
                            movie_country = clean_value

                desc_el = m_soup.find(attrs={"itemprop": "description"})
                description = ' '.join(desc_el.text.replace('\n', ' ').split()) if desc_el else "Нет описания"

                result_list['title'].append(title)
                result_list['year'].append(year)
                result_list['rating'].append(rating)
                result_list['genre'].append(movie_genre)
                result_list['country'].append(movie_country)
                result_list['description'].append(description)

                count += 1
                print(f"[{count}] {title} | {movie_genre} | {movie_country}")
                
            except Exception: continue
            time.sleep(1)
        pagenum += 1
    except Exception: break

df = pd.DataFrame(result_list)
df.to_csv('zona_data.csv', index=False, encoding='utf-8-sig')
print("\n Файл 'zona_clean_data.csv' готов")

[1] Ограбление в Лос-Анджелесе | криминал, драма, триллер | США , Великобритания
[2] Удачи, веселья, не сдохни | боевик, драма, комедия, фантастика | США , Германия
[3] Острые козырьки: Бессмертный человек | история, драма, криминал | США , Франция , Великобритания
[4] Чебурашка 2 | фэнтези, комедия, семейный | Россия
[5] Король и Шут. Навсегда | музыка, приключения, драма, комедия, фэнтези | Россия
[6] Свои. Баллада о войне | драма, военный | Россия
[7] Аватар: Пламя и пепел | приключения, боевик, фэнтези, фантастика | Канада , США
[8] Левша | детектив, приключения, фантастика | Россия
[9] Крик 7 | триллер, ужасы | Канада , США
[10] Казнить нельзя помиловать | фантастика, боевик, триллер, детектив | Россия , США
[11] Грозовой перевал | драма, мелодрама | США , Великобритания
[12] Проект «Конец света» | драма, триллер, фантастика | США
[13] Три богатыря и свет клином | семейный, приключения, фэнтези, комедия, мультфильм | Россия
[14] Доспехи | приключения, комедия, драма, боевик, фэнте

In [11]:
df

,title,genre,year,country,rating,description
0,Ограбление в Лос-Анджелесе,"криминал, драма, триллер",2026,"США , Великобритания",6.7,Дерзкие ограбления неуловимого вора Дэвиса вдо...
1,"Удачи, веселья, не сдохни","боевик, драма, комедия, фантастика",2025,"США , Германия",7.0,Одним вечером в обычной закусочной появляется ...
2,Острые козырьки: Бессмертный человек,"история, драма, криминал",2026,"США , Франция , Великобритания",6.7,"1940-е годы, разгар Второй мировой войны. Спус..."
3,Чебурашка 2,"фэнтези, комедия, семейный",2025,Россия,7.6,"Уже год, как Чебурашка живет у Гены. Ушастик в..."
4,Король и Шут. Навсегда,"музыка, приключения, драма, комедия, фэнтези",2026,Россия,6.4,Панк-вселенной «Короля и Шута» грозит гибель. ...
...,...,...,...,...,...,...
245,Человек-паук: Паутина вселенных,"семейный, приключения, боевик, фэнтези, фантас...",2023,США,8.4,После воссоединения с Гвен Стейси дружелюбный ...
246,Форсаж 6,"криминал, драма, триллер, боевик",2013,"Испания , Япония , США",7.0,"После того как Доминик и Брайн побывали в Рио,..."
247,Исчезнувшая,"детектив, драма, триллер",2014,США,8.0,Всё было готово для празднования пятилетия суп...
248,Падение Луны,"боевик, фантастика",2022,"Канада , Китай , США",6.3,В 2011 году во время рядового ремонта спутника...


In [12]:
print("Количество нулевых значений в: ")
for i in result_list:
    print( i + " - " + str(result_list[i].count(None)))

Количество нулевых значений в: 
title - 0
genre - 0
year - 0
country - 0
rating - 0
description - 0


# Парсинг с API

In [15]:
api_token = "M7HKBXS-FWH4JZB-NP91GN4-P0XEMSD"
api_url = "https://api.poiskkino.dev/v1/movie"

In [16]:
api_headers = {
    "X-API-KEY": api_token,
    "Content-Type": "application/json"
}

In [17]:
api_movies = {
    'title': [], 
    'genre': [], 
    'year': [], 
    'country': [], 
    'rating': [], 
    'description': []
}

In [18]:
for page in range(1, 6):
    params = {
        "page": page,
        "limit": 50,
        "selectFields": ["name", "year", "rating", "genres", "countries", "description"]
    }
    
    try:
        response = requests.get(api_url, headers=api_headers, params=params, timeout=10)
        data = response.json()
        
        if 'docs' in data:
            for item in data['docs']:
                name = item.get('name') or item.get('alternativeName') or "Без названия"
                
                genres_list = [g.get('name') for g in item.get('genres', [])]
                countries_list = [c.get('name') for c in item.get('countries', [])]
                
                api_movies['title'].append(name)
                api_movies['genre'].append(", ".join(genres_list) if genres_list else "Не указан")
                api_movies['year'].append(item.get('year', "N/A"))
                api_movies['country'].append(", ".join(countries_list) if countries_list else "Не указана")
                
                rating_val = item.get('rating', {})
                if isinstance(rating_val, dict):
                    api_movies['rating'].append(rating_val.get('kp', 0))
                else:
                    api_movies['rating'].append(rating_val)
                    
                api_movies['description'].append(item.get('description', "Нет описания"))
            
            print(f"API: Спарсили страницу {page} (всего {len(api_movies['title'])} фильмов)")
        else:
            print(f"Ошибка в структуре ответа API на странице {page}")
            
    except Exception as e:
        print(f"Ошибка при запросе к API: {e}")
    
    time.sleep(0.5)

df_api = pd.DataFrame(api_movies)
combined_df = pd.concat([df, df_api], ignore_index=True)
combined_df.drop_duplicates(subset=['title', 'year'], keep='first', inplace=True)
combined_df.to_csv('total_500_movies.csv', index=False, encoding='utf-8-sig')
print(f"После удаления дублей осталось фильмов: {len(combined_df)}")

API: Спарсили страницу 1 (всего 50 фильмов)
API: Спарсили страницу 2 (всего 100 фильмов)
API: Спарсили страницу 3 (всего 150 фильмов)
API: Спарсили страницу 4 (всего 200 фильмов)
API: Спарсили страницу 5 (всего 250 фильмов)
После удаления дублей осталось фильмов: 500


In [20]:
print("Количество нулевых значений в: ")
for i in api_movies:
    print( i + " - " + str(api_movies[i].count(None)))

Количество нулевых значений в: 
title - 0
genre - 0
year - 0
country - 0
rating - 0
description - 0


In [19]:
combined_df

,title,genre,year,country,rating,description
0,Ограбление в Лос-Анджелесе,"криминал, драма, триллер",2026,"США , Великобритания",6.7,Дерзкие ограбления неуловимого вора Дэвиса вдо...
1,"Удачи, веселья, не сдохни","боевик, драма, комедия, фантастика",2025,"США , Германия",7.0,Одним вечером в обычной закусочной появляется ...
2,Острые козырьки: Бессмертный человек,"история, драма, криминал",2026,"США , Франция , Великобритания",6.7,"1940-е годы, разгар Второй мировой войны. Спус..."
3,Чебурашка 2,"фэнтези, комедия, семейный",2025,Россия,7.6,"Уже год, как Чебурашка живет у Гены. Ушастик в..."
4,Король и Шут. Навсегда,"музыка, приключения, драма, комедия, фэнтези",2026,Россия,6.4,Панк-вселенной «Короля и Шута» грозит гибель. ...
...,...,...,...,...,...,...
495,Список Шиндлера,"биография, история, драма, военный",1993,США,8.851,"Оскар Шиндлер, член национал-социалистической ..."
496,Гордость и предубеждение,"драма, мелодрама",2005,"Великобритания, Франция, США",8.151,"Англия, конец XVIII века. Родители пятерых сес..."
497,Отряд самоубийц,"фантастика, боевик, фэнтези, приключения",2016,"США, Канада",6.104,Правительство решает дать команде суперзлодеев...
498,Постучись в мою Тверь,"комедия, мелодрама",2024,Россия,7.086,"Городская фифа Алиса приезжает в Тверь, чтобы ..."
